In [1]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="experiment_1c",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

In [2]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

MODEL = "gpt-4.1-mini"

LABEL_MAP = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}
ALLOWED_LABELS = list(LABEL_MAP.values())


---[ TableVault Record ]---
---[ TableVault Record ]---



In [3]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
data = vault.query_item_content("ag_news_small")
df = pd.DataFrame(data)

df.head()


---[ TableVault Record ]---


,text,label
0,Indian board plans own telecast of Australia s...,1
1,Stocks Higher on Drop in Jobless Claims A shar...,2
2,"Nuggets 112, Raptors 106 Carmelo Anthony score...",1
3,Stocks Higher on Drop in Jobless Claims A shar...,2
4,REVIEW: 'Half-Life 2' a Tech Masterpiece (AP) ...,3


---[ TableVault Record ]---



In [5]:
FEW_SHOT_EXAMPLES = [
    {
        "text": "The United Nations held emergency talks after rising tensions between two neighboring countries.",
        "label": "World",
        "reason": "This is about international affairs and diplomacy."
    },
    {
        "text": "The star forward scored twice to lead the team to a 3-1 victory in the finals.",
        "label": "Sports",
        "reason": "This is about a sports match and player performance."
    },
    {
        "text": "Shares rose after the company reported stronger-than-expected quarterly earnings.",
        "label": "Business",
        "reason": "This is about company earnings and markets."
    },
    {
        "text": "Researchers unveiled a new AI model that improves medical image analysis.",
        "label": "Sci/Tech",
        "reason": "This is about scientific/technical research and AI."
    },
]


---[ TableVault Record ]---
---[ TableVault Record ]---



In [6]:
def build_few_shot_prompt(text: str) -> str:
    example_blocks = []
    for ex in FEW_SHOT_EXAMPLES:
        example_blocks.append(
            f"""Example:
Text: {ex['text']}
Label: {ex['label']}
Why: {ex['reason']}"""
        )

    examples_text = "\n\n".join(example_blocks)

    return f"""
You are a careful news classifier.

Possible labels:
- World
- Sports
- Business
- Sci/Tech

Use the examples below to understand the labels.

{examples_text}

Now classify this new article.

Return strict JSON with keys:
- predicted_label
- short_reason

The predicted_label must be exactly one of:
["World", "Sports", "Business", "Sci/Tech"]

Do not use markdown code fences.

Article:
{text}
""".strip()


---[ TableVault Record ]---
---[ TableVault Record ]---



In [7]:
def extract_first_json_object(raw_text: str) -> dict:
    if not raw_text or not raw_text.strip():
        raise ValueError("Model returned empty output.")

    decoder = json.JSONDecoder()
    for i, ch in enumerate(raw_text):
        if ch != "{":
            continue
        try:
            candidate, _ = decoder.raw_decode(raw_text[i:])
        except json.JSONDecodeError:
            continue
        if isinstance(candidate, dict):
            return candidate

    raise ValueError(f"Could not find JSON object in model output: {raw_text!r}")


def classify_few_shot(text: str) -> dict:
    prompt = build_few_shot_prompt(text)

    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )

    raw = response.output_text or ""
    print(raw)
    result = extract_first_json_object(raw)

    if result.get("predicted_label") not in ALLOWED_LABELS:
        raise ValueError(f"Unexpected label: {result.get('predicted_label')}")

    if not isinstance(result.get("short_reason"), str):
        raise ValueError("Missing or invalid 'short_reason' in model output.")

    return {
        "predicted_label": result["predicted_label"],
        "short_reason": result["short_reason"],
    }


---[ TableVault Record ]---
---[ TableVault Record ]---



In [8]:
vault.create_record_list("few_shot_output", column_names=["gold_label", "predicted_label", "short_reason"])
description = (
    "Experiment 1c row-level few-shot AG News classification results from ag_news_small. Each record corresponds to one source article and stores gold_label as the ground-truth topic, predicted_label as the GPT-4.1-mini predicted topic, and short_reason as the model explanation. Gold labels follow the AG News mapping World, Sports, Business, and Sci/Tech. The classifier reads up to the first 1000 characters of each article and uses four hand-written few-shot examples. Use this list for article-level evaluation, misclassification search, confusion analysis, rationale inspection, and tracing predictions back to the original ag_news_small record through input_items."
)
embedding = get_embeddings(description)
vault.create_description("few_shot_output", description, embedding)

for i, row in tqdm(df.iterrows(), total=len(df)):
    text = row["text"][:1000]
    pred = classify_few_shot(text)
    input_items = {"ag_news_small": [i, i + 1]}
    vault.append_record("few_shot_output",
                        {
                            "gold_label": LABEL_MAP[row["label"]],
                            "predicted_label": pred["predicted_label"],
                            "short_reason": pred["short_reason"],
                        },
                        input_items = input_items
                       )


---[ TableVault Record ]---


  0%|          | 0/10 [00:00<?, ?it/s]

{
  "predicted_label": "Sports",
  "short_reason": "The article discusses cricket and broadcasting arrangements for a test series, which relates to sports."
}
{
  "predicted_label": "Business",
  "short_reason": "The article discusses stock movements and company forecasts, which relate to financial markets and business."
}
{
  "predicted_label": "Sports",
  "short_reason": "This article reports on a basketball game and player performances."
}
{
  "predicted_label": "Business",
  "short_reason": "Article discusses stock movements influenced by economic data and company forecasts."
}
{
  "predicted_label": "Sci/Tech",
  "short_reason": "The article discusses a video game and its technological and storytelling advancements."
}
{
  "predicted_label": "Business",
  "short_reason": "The article discusses inflation rates and economic conditions in China, which relates to business and economic matters."
}
{"predicted_label":"Business","short_reason":"The article discusses currency trading, lev

In [9]:
results_df = pd.DataFrame(vault.query_item_content("few_shot_output"))
results_df["correct"] = results_df["gold_label"] == results_df["predicted_label"]
accuracy = results_df["correct"].mean()
vault.create_record_list("experiment_1c_accuracy", column_names=["accuracy"])
description = (
    "Experiment 1c aggregate accuracy summary for the few-shot AG News classifier on ag_news_small. This list stores the overall exact-match accuracy in the accuracy column, computed as the fraction of few_shot_output records where gold_label equals predicted_label. It summarizes row-level predictions produced by GPT-4.1-mini for the four AG News topics World, Sports, Business, and Sci/Tech. Use this list for experiment comparison, benchmark reporting, model selection, and quick retrieval of the headline evaluation metric. The record links back to the full few_shot_output range through input_items."
)
embedding = get_embeddings(description)
vault.create_description("experiment_1c_accuracy", description, embedding)



---[ TableVault Record ]---
---[ TableVault Record ]---



In [10]:
input_items = {"few_shot_output": [0, len(results_df)]}
vault.append_record("experiment_1c_accuracy", {"accuracy": accuracy}, input_items=input_items)
accuracy


---[ TableVault Record ]---


np.float64(0.9)

---[ TableVault Record ]---

